In [2]:
import pandas as panda

historical = panda.read_csv(
    "../data/processed/historical.csv",
    parse_dates=["Date"]
)
historical = historical.sort_values("Date").reset_index(drop=True)

historical = historical.dropna(
    subset=[
        "Date",
        "HomeTeam",
        "AwayTeam",
        "FTHG",
        "FTAG",
        "FTR",
        "Season"
    ]
).copy()

teams = sorted(
    set(historical["HomeTeam"]) | set(historical["AwayTeam"])
)

initialElo = 1500.0
homeAdvantage = 60.0
eloK = 40.0

from collections import defaultdict, deque
#maxlen means only the last 5 items are stored
#deque allows the oldest item to be removed if a new item is added automatically
#lambda tells the program that it 
team_history = defaultdict(lambda: {
    # Overall form
    "results": deque(maxlen=5),
    "goals_scored": deque(maxlen=5),
    "goals_conceded": deque(maxlen=5),

    # Home form
    "home_results": deque(maxlen=5),
    "home_goals_scored": deque(maxlen=5),
    "home_goals_conceded": deque(maxlen=5),

    # Away form
    "away_results": deque(maxlen=5),
    "away_goals_scored": deque(maxlen=5),
    "away_goals_conceded": deque(maxlen=5),

    # Current season record
    "season_points": 0,
    "season_games": 0,
    "season_goals_scored": 0,
    "season_goals_conceded": 0,

    # Elo
    "elo": initialElo
})

def weighted_points_per_game(results):
    points = {
        "W": 3,
        "D": 1,
        "L": 0
    }

    weights = list(range(1, len(results) + 1))

    if not weights:
        return 0.0

    weighted_points = sum(
        points[result] * weight
        for result, weight in zip(results, weights)
    )

    return weighted_points / sum(weights)

features = []
current_season = None
for _, match in historical.iterrows():

    season = match["Season"]
    # Reset league-table statistics when a new season begins.
    if season != current_season:
        current_season = season

        for history in team_history.values():
            history["season_points"] = 0
            history["season_games"] = 0
            history["season_goals_scored"] = 0
            history["season_goals_conceded"] = 0

    home = match["HomeTeam"]
    away = match["AwayTeam"]

    home_history = team_history[home]
    away_history = team_history[away]

    # Gets the last 5 results from each team
    home_results = list(home_history["results"])
    away_results = list(away_history["results"])

    # Gets the last 5 home results for the home team and the last 5 away results for the away team
    home_home_results = list(home_history["home_results"])
    away_away_results = list(away_history["away_results"])

    feature_row = {
        "Date": match["Date"],
        "HomeTeam": home,
        "AwayTeam": away,
        "FTR": match["FTR"],

        # Overall form
        "HomeRecentWins": home_results.count("W"),
        "HomeRecentDraws": home_results.count("D"),
        "HomeRecentLosses": home_results.count("L"),
        "HomeRecentGames": len(home_results),

        "AwayRecentWins": away_results.count("W"),
        "AwayRecentDraws": away_results.count("D"),
        "AwayRecentLosses": away_results.count("L"),
        "AwayRecentGames": len(away_results),

        "HomeRecentGoalsScored": sum(
            home_history["goals_scored"]
        ),

        "HomeRecentGoalsConceded": sum(
            home_history["goals_conceded"]
        ),

        "AwayRecentGoalsScored": sum(
            away_history["goals_scored"]
        ),

        "AwayRecentGoalsConceded": sum(
            away_history["goals_conceded"]
        ),
        
        "HomeSeasonPoints": home_history["season_points"],
        "AwaySeasonPoints": away_history["season_points"],

        "HomeSeasonGames": home_history["season_games"],
        "AwaySeasonGames": away_history["season_games"],

        "HomeSeasonGoalDifference": (
            home_history["season_goals_scored"] -
            home_history["season_goals_conceded"]
        ),
        "AwaySeasonGoalDifference": (
            away_history["season_goals_scored"] -
            away_history["season_goals_conceded"]
        ),

        "HomeSeasonPointsPerGame": (
            home_history["season_points"] / home_history["season_games"]
            if home_history["season_games"] > 0 else 0
        ),
        "AwaySeasonPointsPerGame": (
            away_history["season_points"] / away_history["season_games"]
            if away_history["season_games"] > 0 else 0
        ),

        # Home team's home form
        "HomeRecentHomeWins": home_home_results.count("W"),
        "HomeRecentHomeDraws": home_home_results.count("D"),
        "HomeRecentHomeLosses": home_home_results.count("L"),
        "HomeRecentHomeGames": len(home_home_results),

        "HomeRecentHomeGoalsScored": sum(
            home_history["home_goals_scored"]
        ),
        "HomeRecentHomeGoalsConceded": sum(
            home_history["home_goals_conceded"]
        ),

        # Away team's away form
        "AwayRecentAwayWins": away_away_results.count("W"),
        "AwayRecentAwayDraws": away_away_results.count("D"),
        "AwayRecentAwayLosses": away_away_results.count("L"),
        "AwayRecentAwayGames": len(away_away_results),

        "AwayRecentAwayGoalsScored": sum(
            away_history["away_goals_scored"]
        ),
        
        "AwayRecentAwayGoalsConceded": sum(
            away_history["away_goals_conceded"]
        ),

        # Home and Away elo
        "HomeElo": home_history["elo"],
        "AwayElo": away_history["elo"],

        # Weighted form
        "HomeWeightedForm": weighted_points_per_game(home_results),
        "AwayWeightedForm": weighted_points_per_game(away_results),
        "HomeWeightedHomeForm": weighted_points_per_game(home_home_results),
        "AwayWeightedAwayForm": weighted_points_per_game(away_away_results)
    }

    # Save the feature (fixture) row
    features.append(feature_row)
    # Determine outocme of game
    if match["FTR"] == "H":
        home_result = "W"
        away_result = "L"
        home_points = 3
        away_points = 0
        home_elo_score = 1.0
    elif match["FTR"] == "D":
        home_result = "D"
        away_result = "D"
        home_points = 1
        away_points = 1
        home_elo_score = 0.5
    elif match["FTR"] == "A":
        home_result = "L"
        away_result = "W"
        home_points = 0
        away_points = 3
        home_elo_score = 0.0
    else:
        raise ValueError(
        f'Unexpected FTR value: {match["FTR"]}'
        )

    # Update the overall form of both teams    
    home_history["results"].append(home_result)
    away_history["results"].append(away_result)

    home_history["goals_scored"].append(match["FTHG"])
    home_history["goals_conceded"].append(match["FTAG"])

    away_history["goals_scored"].append(match["FTAG"])
    away_history["goals_conceded"].append(match["FTHG"])

    # Home-specific
    home_history["home_results"].append(home_result)
    home_history["home_goals_scored"].append(match["FTHG"])
    home_history["home_goals_conceded"].append(match["FTAG"])

    # Away-specific
    away_history["away_results"].append(away_result)
    away_history["away_goals_scored"].append(match["FTAG"])
    away_history["away_goals_conceded"].append(match["FTHG"])

    home_history["season_points"] += home_points
    away_history["season_points"] += away_points

    home_history["season_games"] += 1
    away_history["season_games"] += 1

    home_history["season_goals_scored"] += match["FTHG"]
    home_history["season_goals_conceded"] += match["FTAG"]

    away_history["season_goals_scored"] += match["FTAG"]
    away_history["season_goals_conceded"] += match["FTHG"]

    home_elo = home_history["elo"]
    away_elo = away_history["elo"]

    expectedHomeScore = 1 / (1 + 10 ** ((away_elo - (home_elo + homeAdvantage)) / 200))

    elo_change = eloK * (home_elo_score - expectedHomeScore)

    home_history["elo"] += elo_change
    away_history["elo"] -= elo_change

features_df = panda.DataFrame(features)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_column = "FTR"
categorical_columns = [
    "HomeTeam",
    "AwayTeam"
]

numeric_columns = [
    column for column in features_df.columns
    if column not in categorical_columns + ["Date", target_column]
]

feature_columns = categorical_columns + numeric_columns

validation_start = panda.Timestamp("2022-08-01")
validation_end = panda.Timestamp("2027-08-01")

train_data = features_df[
    features_df["Date"] < validation_start
]

test_data = features_df[
    (features_df["Date"] >= validation_start) &
    (features_df["Date"] < validation_end)
]


X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

numeric_processor = Pipeline([
    ("missing_values", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("teams", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ("numbers", numeric_processor, numeric_columns)
])

model = Pipeline([
    ("prepare_data", preprocessor),
    ("predict_result", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

comparison = panda.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": predictions
})

print("Always-home-win baseline:", (y_test == "H").mean())

print("\nPrediction counts:")
print(comparison["Predicted"].value_counts())

print("\nConfusion matrix:")
print(panda.crosstab(
    comparison["Actual"],
    comparison["Predicted"],
    rownames=["Actual"],
    colnames=["Predicted"]
))

model.fit(
    features_df[feature_columns],
    features_df[target_column]
)

def predict_fixture(home_team, away_team):
    if home_team not in team_history:
        raise ValueError(f"Unknown home team: {home_team}")

    if away_team not in team_history:
        raise ValueError(f"Unknown away team: {away_team}")

    home_history = team_history[home_team]
    away_history = team_history[away_team]

    home_results = list(home_history["results"])
    away_results = list(away_history["results"])
    home_home_results = list(home_history["home_results"])
    away_away_results = list(away_history["away_results"])

    fixture_features = {
        "HomeTeam": home_team,
        "AwayTeam": away_team,

        "HomeRecentWins": home_results.count("W"),
        "HomeRecentDraws": home_results.count("D"),
        "HomeRecentLosses": home_results.count("L"),
        "HomeRecentGames": len(home_results),

        "AwayRecentWins": away_results.count("W"),
        "AwayRecentDraws": away_results.count("D"),
        "AwayRecentLosses": away_results.count("L"),
        "AwayRecentGames": len(away_results),

        "HomeRecentGoalsScored": sum(home_history["goals_scored"]),
        "HomeRecentGoalsConceded": sum(home_history["goals_conceded"]),
        "AwayRecentGoalsScored": sum(away_history["goals_scored"]),
        "AwayRecentGoalsConceded": sum(away_history["goals_conceded"]),

        "HomeSeasonPoints": home_history["season_points"],
        "AwaySeasonPoints": away_history["season_points"],
        "HomeSeasonGames": home_history["season_games"],
        "AwaySeasonGames": away_history["season_games"],

        "HomeSeasonGoalDifference": (
            home_history["season_goals_scored"] -
            home_history["season_goals_conceded"]
        ),
        "AwaySeasonGoalDifference": (
            away_history["season_goals_scored"] -
            away_history["season_goals_conceded"]
        ),

        "HomeSeasonPointsPerGame": (
            home_history["season_points"] / home_history["season_games"]
            if home_history["season_games"] > 0 else 0
        ),
        "AwaySeasonPointsPerGame": (
            away_history["season_points"] / away_history["season_games"]
            if away_history["season_games"] > 0 else 0
        ),

        "HomeRecentHomeWins": home_home_results.count("W"),
        "HomeRecentHomeDraws": home_home_results.count("D"),
        "HomeRecentHomeLosses": home_home_results.count("L"),
        "HomeRecentHomeGames": len(home_home_results),
        "HomeRecentHomeGoalsScored": sum(
            home_history["home_goals_scored"]
        ),
        "HomeRecentHomeGoalsConceded": sum(
            home_history["home_goals_conceded"]
        ),

        "AwayRecentAwayWins": away_away_results.count("W"),
        "AwayRecentAwayDraws": away_away_results.count("D"),
        "AwayRecentAwayLosses": away_away_results.count("L"),
        "AwayRecentAwayGames": len(away_away_results),
        "AwayRecentAwayGoalsScored": sum(
            away_history["away_goals_scored"]
        ),
        "AwayRecentAwayGoalsConceded": sum(away_history["away_goals_conceded"]),

        "HomeElo": home_history["elo"],
        "AwayElo": away_history["elo"],

        "HomeElo": home_history["elo"],
        "AwayElo": away_history["elo"],

        "HomeWeightedForm": weighted_points_per_game(home_results),
        "AwayWeightedForm": weighted_points_per_game(away_results),

        "HomeWeightedHomeForm": weighted_points_per_game(home_home_results),
        "AwayWeightedAwayForm": weighted_points_per_game(away_away_results)
    }

    fixture_df = panda.DataFrame([fixture_features])
    fixture_df = fixture_df[feature_columns]

    probabilities = model.predict_proba(fixture_df)[0]

    return dict(zip(model.classes_, probabilities))


print(
    features_df[
        ["HomeElo", "AwayElo"]
    ].describe()
)

Accuracy: 0.46496815286624205
              precision    recall  f1-score   support

           A       0.42      0.60      0.50       494
           D       0.25      0.19      0.22       382
           H       0.62      0.52      0.57       694

    accuracy                           0.46      1570
   macro avg       0.43      0.44      0.43      1570
weighted avg       0.47      0.46      0.46      1570

Always-home-win baseline: 0.44203821656050957

Prediction counts:
Predicted
A    703
H    582
D    285
Name: count, dtype: int64

Confusion matrix:
Predicted    A    D    H
Actual                  
A          297   93  104
D          193   72  117
H          213  120  361
           HomeElo      AwayElo
count  8790.000000  8790.000000
mean   1542.921822  1540.953519
std      72.766996    72.922946
min    1341.247685  1335.708609
25%    1489.141811  1487.442191
50%    1534.396219  1531.306084
75%    1590.312345  1588.623651
max    1806.573962  1806.969315


In [6]:
upcoming_fixtures = [
    # Matchweek 6
    ("Arsenal", "Leeds"),
    ("Aston Villa", "Brentford"),
    ("Chelsea", "Bournemouth"),
    ("Ipswich", "Fulham"),
    ("Sunderland", "Brighton"),
    ("Man United", "Tottenham"),
    ("Crystal Palace", "Nott'm Forest"),
    ("Hull", "Everton"),
    ("Liverpool", "Man City"),
    ("Coventry", "Newcastle"),

    # Matchweek 7
    ("Everton", "Chelsea"),
    ("Brentford", "Liverpool"),
    ("Fulham", "Hull"),
    ("Man City", "Ipswich"),
    ("Newcastle", "Aston Villa"),
    ("Bournemouth", "Sunderland"),
    ("Brighton", "Crystal Palace"),
    ("Leeds", "Man United"),
    ("Nott'm Forest", "Arsenal"),
    ("Tottenham", "Coventry"),

    # Matchweek 8
    ("Ipswich", "Nott'm Forest"),
    ("Aston Villa", "Man City"),
    ("Arsenal", "Everton"),
    ("Coventry", "Fulham"),
    ("Chelsea", "Tottenham"),
    ("Crystal Palace", "Newcastle"),
    ("Hull", "Brentford"),
    ("Liverpool", "Brighton"),
    ("Man United", "Bournemouth"),
    ("Sunderland", "Leeds"),

    # Matchweek 9
    ("Chelsea", "Man United"),
    ("Bournemouth", "Leeds"),
    ("Brentford", "Nott'm Forest"),
    ("Coventry", "Sunderland"),
    ("Hull", "Ipswich"),
    ("Man City", "Brighton"),
    ("Tottenham", "Crystal Palace"),
    ("Aston Villa", "Fulham"),
    ("Liverpool", "Arsenal"),
    ("Newcastle", "Everton"),

    # Matchweek 10
    ("Everton", "Coventry"),
    ("Leeds", "Tottenham"),
    ("Arsenal", "Hull"),
    ("Fulham", "Newcastle"),
    ("Nott'm Forest", "Man City"),
    ("Brighton", "Brentford"),
    ("Crystal Palace", "Liverpool"),
    ("Ipswich", "Bournemouth"),
    ("Sunderland", "Chelsea"),
    ("Man United", "Aston Villa"),

    # Matchweek 11
    ("Man City", "Fulham"),
    ("Aston Villa", "Sunderland"),
    ("Bournemouth", "Nott'm Forest"),
    ("Chelsea", "Leeds"),
    ("Coventry", "Crystal Palace"),
    ("Tottenham", "Ipswich"),
    ("Newcastle", "Arsenal"),
    ("Hull", "Brighton"),
    ("Liverpool", "Man United"),
    ("Brentford", "Everton"),

    # Matchweek 12
    ("Nott'm Forest", "Chelsea"),
    ("Leeds", "Coventry"),
    ("Man United", "Brentford"),
    ("Ipswich", "Aston Villa"),
    ("Everton", "Liverpool"),
    ("Brighton", "Newcastle"),
    ("Crystal Palace", "Hull"),
    ("Fulham", "Bournemouth"),
    ("Sunderland", "Tottenham"),
    ("Arsenal", "Man City"),

    # Matchweek 13
    ("Aston Villa", "Everton"),
    ("Bournemouth", "Brighton"),
    ("Brentford", "Arsenal"),
    ("Chelsea", "Crystal Palace"),
    ("Coventry", "Ipswich"),
    ("Hull", "Nott'm Forest"),
    ("Liverpool", "Sunderland"),
    ("Man City", "Leeds"),
    ("Newcastle", "Man United"),
    ("Tottenham", "Fulham"),

    # Matchweek 14
    ("Aston Villa", "Crystal Palace"),
    ("Bournemouth", "Hull"),
    ("Brentford", "Man City"),
    ("Chelsea", "Liverpool"),
    ("Everton", "Fulham"),
    ("Leeds", "Ipswich"),
    ("Man United", "Coventry"),
    ("Newcastle", "Sunderland"),
    ("Nott'm Forest", "Brighton"),
    ("Tottenham", "Arsenal"),

    # Matchweek 15
    ("Arsenal", "Bournemouth"),
    ("Brighton", "Everton"),
    ("Coventry", "Aston Villa"),
    ("Crystal Palace", "Man United"),
    ("Fulham", "Brentford"),
    ("Hull", "Tottenham"),
    ("Ipswich", "Newcastle"),
    ("Liverpool", "Leeds"),
    ("Man City", "Chelsea"),
    ("Sunderland", "Nott'm Forest"),

    # Matchweek 16
    ("Arsenal", "Man United"),
    ("Bournemouth", "Coventry"),
    ("Brentford", "Newcastle"),
    ("Brighton", "Ipswich"),
    ("Chelsea", "Aston Villa"),
    ("Leeds", "Fulham"),
    ("Liverpool", "Tottenham"),
    ("Man City", "Hull"),
    ("Nott'm Forest", "Everton"),
    ("Sunderland", "Crystal Palace"),

    # Matchweek 17
    ("Aston Villa", "Leeds"),
    ("Everton", "Sunderland"),
    ("Fulham", "Brighton"),
    ("Ipswich", "Brentford"),
    ("Tottenham", "Bournemouth"),
    ("Hull", "Liverpool"),
    ("Newcastle", "Man City"),
    ("Coventry", "Chelsea"),
    ("Man United", "Nott'm Forest"),
    ("Crystal Palace", "Arsenal"),

    # Matchweek 18
    ("Hull", "Leeds"),
    ("Tottenham", "Brighton"),
    ("Everton", "Man City"),
    ("Coventry", "Brentford"),
    ("Crystal Palace", "Bournemouth"),
    ("Fulham", "Arsenal"),
    ("Man United", "Sunderland"),
    ("Newcastle", "Nott'm Forest"),
    ("Aston Villa", "Liverpool"),
    ("Ipswich", "Chelsea"),

    # Matchweek 19
    ("Leeds", "Everton"),
    ("Brighton", "Man United"),
    ("Arsenal", "Ipswich"),
    ("Brentford", "Crystal Palace"),
    ("Liverpool", "Coventry"),
    ("Nott'm Forest", "Fulham"),
    ("Sunderland", "Hull"),
    ("Bournemouth", "Aston Villa"),
    ("Chelsea", "Newcastle"),
    ("Man City", "Tottenham"),

    # Matchweek 20
    ("Arsenal", "Brentford"),
    ("Brighton", "Bournemouth"),
    ("Everton", "Aston Villa"),
    ("Nott'm Forest", "Hull"),
    ("Sunderland", "Liverpool"),
    ("Crystal Palace", "Chelsea"),
    ("Fulham", "Tottenham"),
    ("Ipswich", "Coventry"),
    ("Leeds", "Man City"),
    ("Man United", "Newcastle"),

    # Matchweek 21
    ("Aston Villa", "Man United"),
    ("Bournemouth", "Ipswich"),
    ("Brentford", "Brighton"),
    ("Chelsea", "Sunderland"),
    ("Coventry", "Everton"),
    ("Hull", "Arsenal"),
    ("Liverpool", "Crystal Palace"),
    ("Man City", "Nott'm Forest"),
    ("Newcastle", "Fulham"),
    ("Tottenham", "Leeds"),

    # Matchweek 22
    ("Arsenal", "Newcastle"),
    ("Brighton", "Man City"),
    ("Crystal Palace", "Tottenham"),
    ("Everton", "Brentford"),
    ("Fulham", "Aston Villa"),
    ("Ipswich", "Hull"),
    ("Leeds", "Chelsea"),
    ("Man United", "Liverpool"),
    ("Nott'm Forest", "Bournemouth"),
    ("Sunderland", "Coventry"),

    # Matchweek 23
    ("Aston Villa", "Ipswich"),
    ("Bournemouth", "Fulham"),
    ("Brentford", "Man United"),
    ("Chelsea", "Nott'm Forest"),
    ("Coventry", "Leeds"),
    ("Hull", "Crystal Palace"),
    ("Liverpool", "Everton"),
    ("Man City", "Arsenal"),
    ("Newcastle", "Brighton"),
    ("Tottenham", "Sunderland"),

    # Matchweek 24
    ("Arsenal", "Liverpool"),
    ("Brighton", "Hull"),
    ("Crystal Palace", "Coventry"),
    ("Everton", "Newcastle"),
    ("Fulham", "Man City"),
    ("Ipswich", "Tottenham"),
    ("Leeds", "Bournemouth"),
    ("Man United", "Chelsea"),
    ("Nott'm Forest", "Brentford"),
    ("Sunderland", "Aston Villa"),

    # Matchweek 25
    ("Aston Villa", "Bournemouth"),
    ("Coventry", "Liverpool"),
    ("Crystal Palace", "Brentford"),
    ("Everton", "Leeds"),
    ("Fulham", "Nott'm Forest"),
    ("Hull", "Sunderland"),
    ("Ipswich", "Arsenal"),
    ("Man United", "Brighton"),
    ("Newcastle", "Chelsea"),
    ("Tottenham", "Man City"),

    # Matchweek 26
    ("Arsenal", "Fulham"),
    ("Bournemouth", "Crystal Palace"),
    ("Brentford", "Coventry"),
    ("Brighton", "Tottenham"),
    ("Chelsea", "Ipswich"),
    ("Leeds", "Aston Villa"),
    ("Liverpool", "Hull"),
    ("Man City", "Newcastle"),
    ("Nott'm Forest", "Man United"),
    ("Sunderland", "Everton"),

    # Matchweek 27
    ("Aston Villa", "Chelsea"),
    ("Coventry", "Bournemouth"),
    ("Crystal Palace", "Sunderland"),
    ("Everton", "Nott'm Forest"),
    ("Fulham", "Leeds"),
    ("Hull", "Man City"),
    ("Ipswich", "Brighton"),
    ("Man United", "Arsenal"),
    ("Newcastle", "Brentford"),
    ("Tottenham", "Liverpool"),

    # Matchweek 28
    ("Arsenal", "Crystal Palace"),
    ("Bournemouth", "Tottenham"),
    ("Brentford", "Ipswich"),
    ("Brighton", "Fulham"),
    ("Chelsea", "Coventry"),
    ("Leeds", "Hull"),
    ("Liverpool", "Aston Villa"),
    ("Man City", "Everton"),
    ("Nott'm Forest", "Newcastle"),
    ("Sunderland", "Man United"),

    # Matchweek 29
    ("Aston Villa", "Hull"),
    ("Bournemouth", "Newcastle"),
    ("Chelsea", "Arsenal"),
    ("Coventry", "Man City"),
    ("Crystal Palace", "Fulham"),
    ("Leeds", "Brighton"),
    ("Liverpool", "Ipswich"),
    ("Man United", "Everton"),
    ("Sunderland", "Brentford"),
    ("Tottenham", "Nott'm Forest"),

    # Matchweek 30
    ("Arsenal", "Sunderland"),
    ("Brentford", "Bournemouth"),
    ("Brighton", "Coventry"),
    ("Everton", "Tottenham"),
    ("Fulham", "Liverpool"),
    ("Hull", "Chelsea"),
    ("Ipswich", "Crystal Palace"),
    ("Man City", "Man United"),
    ("Newcastle", "Leeds"),
    ("Nott'm Forest", "Aston Villa"),

    # Matchweek 31
    ("Aston Villa", "Brighton"),
    ("Bournemouth", "Man City"),
    ("Chelsea", "Fulham"),
    ("Coventry", "Arsenal"),
    ("Crystal Palace", "Everton"),
    ("Leeds", "Nott'm Forest"),
    ("Liverpool", "Newcastle"),
    ("Man United", "Hull"),
    ("Sunderland", "Ipswich"),
    ("Tottenham", "Brentford"),

    # Matchweek 32
    ("Arsenal", "Aston Villa"),
    ("Brentford", "Leeds"),
    ("Brighton", "Chelsea"),
    ("Everton", "Bournemouth"),
    ("Fulham", "Sunderland"),
    ("Hull", "Coventry"),
    ("Ipswich", "Man United"),
    ("Man City", "Crystal Palace"),
    ("Newcastle", "Tottenham"),
    ("Nott'm Forest", "Liverpool"),

    # Matchweek 33
    ("Aston Villa", "Coventry"),
    ("Bournemouth", "Arsenal"),
    ("Brentford", "Fulham"),
    ("Chelsea", "Man City"),
    ("Everton", "Brighton"),
    ("Leeds", "Liverpool"),
    ("Man United", "Crystal Palace"),
    ("Newcastle", "Ipswich"),
    ("Nott'm Forest", "Sunderland"),
    ("Tottenham", "Hull"),

    # Matchweek 34
    ("Arsenal", "Tottenham"),
    ("Brighton", "Nott'm Forest"),
    ("Coventry", "Man United"),
    ("Crystal Palace", "Aston Villa"),
    ("Fulham", "Everton"),
    ("Hull", "Bournemouth"),
    ("Ipswich", "Leeds"),
    ("Liverpool", "Chelsea"),
    ("Man City", "Brentford"),
    ("Sunderland", "Newcastle"),

    # Matchweek 35
    ("Bournemouth", "Man United"),
    ("Brentford", "Aston Villa"),
    ("Brighton", "Sunderland"),
    ("Everton", "Hull"),
    ("Fulham", "Ipswich"),
    ("Leeds", "Arsenal"),
    ("Man City", "Liverpool"),
    ("Newcastle", "Coventry"),
    ("Nott'm Forest", "Crystal Palace"),
    ("Tottenham", "Chelsea"),

    # Matchweek 36
    ("Arsenal", "Nott'm Forest"),
    ("Aston Villa", "Newcastle"),
    ("Chelsea", "Everton"),
    ("Coventry", "Tottenham"),
    ("Crystal Palace", "Brighton"),
    ("Hull", "Fulham"),
    ("Ipswich", "Man City"),
    ("Liverpool", "Brentford"),
    ("Man United", "Leeds"),
    ("Sunderland", "Bournemouth"),

    # Matchweek 37
    ("Bournemouth", "Chelsea"),
    ("Brentford", "Hull"),
    ("Brighton", "Liverpool"),
    ("Everton", "Arsenal"),
    ("Fulham", "Coventry"),
    ("Leeds", "Sunderland"),
    ("Man City", "Aston Villa"),
    ("Newcastle", "Crystal Palace"),
    ("Nott'm Forest", "Ipswich"),
    ("Tottenham", "Man United"),

    # Matchweek 38
    ("Arsenal", "Brighton"),
    ("Aston Villa", "Tottenham"),
    ("Chelsea", "Brentford"),
    ("Coventry", "Nott'm Forest"),
    ("Crystal Palace", "Leeds"),
    ("Hull", "Newcastle"),
    ("Ipswich", "Everton"),
    ("Liverpool", "Bournemouth"),
    ("Man United", "Fulham"),
    ("Sunderland", "Man City"),
]

predicted_fixtures = []

for home_team, away_team in upcoming_fixtures:
    prediction = predict_fixture(home_team, away_team)

    predicted_fixtures.append({
        "HomeTeam": home_team,
        "AwayTeam": away_team,
        "HomeWin": prediction["H"],
        "Draw": prediction["D"],
        "AwayWin": prediction["A"],
        "PredictedOutcome": max(
            prediction,
            key=prediction.get
        )
    })

predictions_df = panda.DataFrame(predicted_fixtures)

for column in [
    "HomeWin",
    "Draw",
    "AwayWin"
]:
    predictions_df[column] = (
        predictions_df[column] * 100
    ).round(2)

print(predictions_df)

        HomeTeam     AwayTeam  HomeWin   Draw  AwayWin PredictedOutcome
0        Arsenal        Leeds    58.19  25.42    16.39                H
1    Aston Villa    Brentford    24.12  32.05    43.83                A
2        Chelsea  Bournemouth    33.86  36.47    29.66                D
3        Ipswich       Fulham    10.50  30.67    58.84                A
4     Sunderland     Brighton    11.34  43.99    44.67                A
..           ...          ...      ...    ...      ...              ...
325         Hull    Newcastle    25.64  34.63    39.73                A
326      Ipswich      Everton     7.27  31.43    61.30                A
327    Liverpool  Bournemouth    44.97  37.38    17.65                H
328   Man United       Fulham    54.88  28.87    16.25                H
329   Sunderland     Man City    10.45  28.77    60.78                A

[330 rows x 6 columns]
